In [ ]:
import sys
from pathlib import Path

sys.path.append(str(Path.cwd().parent / "src"))
from paths import PROCESSED, RESULTS, savefig
from pypsa_helpers import (
    solve_scenario,
    require_optimal,
    total_co2,
    total_system_cost,
    capacity_by_technology,
    generation_by_technology,
)

import pypsa
import pandas as pd
import matplotlib.pyplot as plt

Scenario 2: 100% CO2 Reduction (Zero Emissions)

Loads the network built by `06_building_the_PyPSA_model_draft.ipynb` and solves it with a
`GlobalConstraint` capping total annual CO2 at 0 tonnes - the existing CCGT fleet gets
dispatched to zero everywhere, and the model must cover all demand from hydro, wind, solar,
battery, H2 storage and transmission instead.

In [ ]:
n = pypsa.Network(str(PROCESSED / "pypsa_network_base.nc"))
cost_scale = n.meta["cost_scale"]

n = solve_scenario(n, co2_limit=0)

In [ ]:
print("status:", n.meta["status"], "| termination condition:", n.meta["termination_condition"])

# Refuse to report numbers from a solve that didn't actually finish: HiGHS's objective
# reads back as 0.0 whenever termination_condition != "optimal" (crossover not run), even
# though p_nom_opt/dispatch are already populated with a not-yet-feasible iterate - that
# looks like "zero cost" or "the CO2 cap didn't work" but is really just an unfinished solve.
require_optimal(n)

total_cost_eur = total_system_cost(n, cost_scale=cost_scale)
total_co2_t = total_co2(n)

print(f"total annual system cost: {total_cost_eur:,.0f} EUR/yr")
print(f"total CO2 emissions: {total_co2_t:,.0f} t/yr")

In [ ]:
n.export_to_netcdf(str(PROCESSED / "pypsa_results_100pct_co2_reduction.nc"))

Built Capacity vs. Actual Generation by Technology

In [ ]:
capacity_table = capacity_by_technology(n)
power_capacity = capacity_table.drop(index=["H2 store", "H2 electrolysis"], errors="ignore")
generation = generation_by_technology(n).reindex(power_capacity.index, fill_value=0)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

power_capacity.plot.bar(ax=axes[0])
axes[0].set_ylabel("Optimal capacity (MW)")
axes[0].set_title("Built Capacity by Technology")

generation.plot.bar(ax=axes[1], color="tab:orange")
axes[1].set_ylabel("Annual generation (MWh/yr)")
axes[1].set_title("Actual Generation by Technology")

fig.suptitle("100% CO2 Reduction")
fig.tight_layout()
savefig(fig, "pypsa/scenarios", "100pct_co2_reduction_capacity.png")
plt.show()